# QLoRA distillation for ontology extraction (R5 / RQ2)

Runs on a **Colab or Kaggle GPU** (T4 16 GB is enough). The laptop has no CUDA (D11).

1. On the laptop: `python -m extraction.distill --make-set 1500`, run the teacher with `extraction.run`, then
   `python -m extraction.distill --export groq__openai__gpt-oss-120b`.
2. Upload `data/processed/distill/train.jsonl` and `val.jsonl` here (Files panel, or a private Kaggle dataset).
   They contain only anonymized Kaggle text (D2).
3. Run all cells. Download `jobx-q4_k_m.gguf` and the `Modelfile`, then on the laptop:
   `ollama create qwen2.5-7b-jobx -f Modelfile` and add `{provider: ollama, model: qwen2.5-7b-jobx}` to
   `benchmark_models` in `config.yaml`.

Base model: **Qwen2.5-7B-Instruct** (open weights, no gated access, and the same base as the zero-shot
`qwen2.5:7b` in the benchmark, so the LoRA gain is measured against its own base too).

In [ ]:
!pip -q install "transformers>=4.44" "peft>=0.12" "trl>=0.10" "accelerate>=0.33" "bitsandbytes>=0.43" datasets

In [ ]:
import torch, json
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer

BASE = "Qwen/Qwen2.5-7B-Instruct"
SEED = 573
torch.manual_seed(SEED)
data = load_dataset("json", data_files={"train": "train.jsonl", "validation": "val.jsonl"})
print(data)

In [ ]:
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
                         bnb_4bit_compute_dtype=torch.float16)
tok = AutoTokenizer.from_pretrained(BASE)
tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, device_map="auto", torch_dtype=torch.float16)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])

In [ ]:
args = SFTConfig(output_dir="out", num_train_epochs=2, per_device_train_batch_size=1, gradient_accumulation_steps=16,
                 learning_rate=2e-4, lr_scheduler_type="cosine", warmup_ratio=0.03, logging_steps=10,
                 eval_strategy="steps", eval_steps=100, save_steps=100, save_total_limit=2, fp16=True,
                 gradient_checkpointing=True, max_length=3072, assistant_only_loss=True, seed=SEED,
                 report_to="none")
trainer = SFTTrainer(model=model, args=args, train_dataset=data["train"], eval_dataset=data["validation"],
                     peft_config=lora, processing_class=tok)
trainer.train()
trainer.save_model("adapter")

If you hit out-of-memory: lower `max_length` to 2048, or switch `BASE` to `Qwen/Qwen2.5-3B-Instruct` (risk plan fallback). Record any change in DECISIONS.md.

In [ ]:
# merge the adapter into fp16 weights, then convert to GGUF (q4_K_M) for Ollama
from peft import AutoPeftModelForCausalLM
merged = AutoPeftModelForCausalLM.from_pretrained("adapter", torch_dtype=torch.float16, device_map="auto").merge_and_unload()
merged.save_pretrained("merged", safe_serialization=True)
tok.save_pretrained("merged")

In [ ]:
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp
!pip -q install -r llama.cpp/requirements.txt
!python llama.cpp/convert_hf_to_gguf.py merged --outfile jobx-f16.gguf --outtype f16
!cmake -S llama.cpp -B llama.cpp/build -DGGML_CUDA=OFF > /dev/null && cmake --build llama.cpp/build --target llama-quantize -j > /dev/null
!llama.cpp/build/bin/llama-quantize jobx-f16.gguf jobx-q4_k_m.gguf Q4_K_M

In [ ]:
open("Modelfile", "w").write("""FROM ./jobx-q4_k_m.gguf
TEMPLATE \"\"\"{{- if .System }}<|im_start|>system
{{ .System }}<|im_end|>
{{ end }}{{- range .Messages }}<|im_start|>{{ .Role }}
{{ .Content }}<|im_end|>
{{ end }}<|im_start|>assistant
\"\"\"
PARAMETER stop <|im_end|>
PARAMETER temperature 0
""")
print("download jobx-q4_k_m.gguf and Modelfile")